In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/clean/ab_clean.csv")
s = df.groupby("group")["converted"].agg(n="size", x="sum")
rng = np.random.default_rng(42)
post_c = rng.beta(1 + s.loc["control", "x"], 1 + s.loc["control", "n"] - s.loc["control", "x"], 200_000)
post_t = rng.beta(1 + s.loc["treatment", "x"], 1 + s.loc["treatment", "n"] - s.loc["treatment", "x"], 200_000)
diff = post_t - post_c

value_per_conversion = 50
users_per_day = len(df) / df["date"].nunique()
annual_users = users_per_day * 365
print(f"Annual users: {annual_users:,.0f}")

Annual users: 4,611,442


In [2]:
ship = diff * annual_users * value_per_conversion
extend_days = 182
extend_cost = diff.mean() * (users_per_day / 2) * extend_days * value_per_conversion

options = pd.DataFrame({
    "option": ["Ship new page", "Keep old page", "Run 6 more months (to detect a 1% lift)"],
    "expected_£_per_year": [ship.mean(), 0, extend_cost],
    "worst_case_£ (2.5%)": [np.percentile(ship, 2.5), 0, np.nan],
    "best_case_£ (97.5%)": [np.percentile(ship, 97.5), 0, np.nan],
    "expected_regret_£": [np.maximum(-diff, 0).mean() * annual_users * value_per_conversion,
                          np.maximum(diff, 0).mean() * annual_users * value_per_conversion, np.nan]})
options.round(0)

,option,expected_£_per_year,worst_case_£ (2.5%),best_case_£ (97.5%),expected_regret_£
0,Ship new page,-363131.0,-906900.0,184099.0,375658.0
1,Keep old page,0.0,0.0,0.0,12526.0
2,Run 6 more months (to detect a 1% lift),-90534.0,NaN,NaN,NaN
